In [ ]:
from darts import TimeSeries

# Target Series with Static Covariates
# Note: In Darts, static covariates are attached to the Target series
all_targets = TimeSeries.from_group_dataframe(
    pandas_df,
    group_cols="PARENT_DEALER_CODE_MODEL_FAMILY",
    value_cols="NET_SALES",
    static_cols=static_cols, # Use the list you defined
    freq='MS'
)

# Future Covariates (only if you have external variables like Price or Holidays)
# If your 'future_covariates' list is empty, Darts' 'add_encoders' will handle 
# the month/year/relative position automatically!
all_future_covs = TimeSeries.from_group_dataframe(
    pandas_df,
    group_cols="PARENT_DEALER_CODE_MODEL_FAMILY",
    value_cols=future_covariates, 
    freq='MS'
) if future_covariates else None

In [ ]:
from darts.dataprocessing.transformers import Scaler
import pandas as pd

# Define cutoff
train_cutoff = pd.Timestamp('2025-10-01')

# 1. Split
train_targets = [s.split_before(train_cutoff)[0] for s in all_targets]
val_targets = [s.split_before(train_cutoff)[1] for s in all_targets]

# 2. Scale Targets (Global Scaling)
# This scales the sales values between 0 and 1
target_scaler = Scaler()
train_targets_transformed = target_scaler.fit_transform(train_targets)
val_targets_transformed = target_scaler.transform(val_targets)

# 3. Scaling Static Covariates (Numeric only)
# Since you label encoded, these are now numbers. 
# While TFT uses embeddings for categories, scaling helps the optimizer.
from darts.dataprocessing.transformers import StaticCovariatesTransformer
static_scaler = StaticCovariatesTransformer()
train_targets_transformed = static_scaler.fit_transform(train_targets_transformed)
val_targets_transformed = static_scaler.transform(val_targets_transformed)

In [ ]:
from darts.models import TFTModel

# Calculate the cardinality (number of unique values) for each categorical column
# The model needs this to size the "embedding" brain correctly
categorical_embedding_sizes = {
    col: len(encoders[col].classes_) for col in static_cols
}

model = TFTModel(
    input_chunk_length=12,
    output_chunk_length=6,
    hidden_size=64,
    lstm_layers=2,
    num_attention_heads=4,
    dropout=0.1,
    batch_size=256,
    n_epochs=50,
    
    # CRITICAL: Tell TFT which static cols are categorical
    categorical_static_covariates=static_cols,
    
    # Pass your pre-defined encoders (cyclic month, etc.)
    add_encoders=add_encoders,
    
    # PyTorch Lightning kwargs for performance
    pl_trainer_kwargs={"accelerator": "gpu", "devices": [0]} # Use GPU if available!
)

In [ ]:
model.fit(
    series=train_targets_transformed,
    val_series=val_targets_transformed,
    future_covariates=all_future_covs_transformed if all_future_covs else None,
    verbose=True
)